# Robson Classification and C-Section Predictive System
## Results Presentation Notebook

**Purpose:** This notebook loads pre-computed results from CSV files and displays them. It requires no heavy data files and opens in seconds.

**For the committee:** Every table shown here was produced by a named Python script from a named input. The six documentation notebooks (01–06) contain the full code with explanations. This notebook shows only the results.

---

### Setup

Place all CSV output files in the same folder as this notebook, or adjust the `DATA` path below.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

# Adjust this path to where your CSV output files are located
DATA = Path(".")

def load(name):
    """Load a CSV, trying several common locations."""
    for folder in [DATA, DATA/"05_outputs", DATA/"csv_export", DATA/"dashboard_data"]:
        p = folder / name
        if p.exists():
            return pd.read_csv(p)
    print(f"  [not found] {name}")
    return None

print("Ready.")

---

## 1. The Reference Population — CDC 2023

3,656,914 births classified into Robson groups. External validation:

| Metric | Our result | Published | Source |
|--------|-----------|-----------|--------|
| Overall CS rate | **32.35%** | 32.3% | NCHS, Final Data 2023 |
| NTSV (Groups 1+2) | **26.56%** | ~26% | ACOG / NCHS |
| Unclassifiable | **0.55%** | <1% threshold | — |

In [ ]:
cdc = load("01_cdc_2023_robson_report.csv")
if cdc is not None:
    print("=" * 80)
    print("WHO STANDARD ROBSON REPORT TABLE — CDC 2023 (3.6 million births)")
    print("=" * 80)
    total = cdc[cdc['robson_group'] != 'TOTAL'] if 'TOTAL' in cdc['robson_group'].values else cdc
    print(cdc.to_string(index=False))
    print()
    if 'n_women' in cdc.columns:
        t = cdc[cdc['robson_group'] != 'TOTAL']
        n = t['n_women'].sum()
        cs = t['n_cesarean'].sum()
        print(f"  Total births: {n:,.0f}")
        print(f"  Total CS:     {cs:,.0f}")
        print(f"  CS rate:      {100*cs/n:.2f}%")

---

## 2. The Egyptian Hospitals

Two hospitals with complete Robson tables — the only two published Egyptian studies with group-level data.

| | Benha 2018 | Qena 2022-23 | CDC 2023 |
|---|---|---|---|
| n | 850 | 2,186 | 3,656,914 |
| CS rate | **54.71%** | **79.69%** | 32.35% |
| Group 5 share | 36.2% | 40.8% | 12.5% |

In [ ]:
benha = load("02_egypt_benha_2018_robson_report.csv")
qena = load("03_egypt_qena_2022_robson_report.csv")

if benha is not None:
    print("BENHA UNIVERSITY HOSPITAL 2018 (n=850)")
    print(benha.to_string(index=False))
    print()

if qena is not None:
    print("QENA (SOUTH VALLEY) 2022-23 (n=2,186)")
    print(qena.to_string(index=False))

In [ ]:
comp = load("04_three_way_comparison__1_.csv")
if comp is None:
    comp = load("04_three_way_comparison.csv")
if comp is not None:
    print("=" * 90)
    print("THREE-WAY COMPARISON: CDC vs Benha vs Qena")
    print("=" * 90)
    print(comp.to_string(index=False))

---

## 3. The Gap — Indirect Standardisation (8-Category Method)

### The three corrections that produced these numbers

1. **8 categories instead of 10** — onset-of-labour groups merged (1+2, 3+4) because the split reflects clinical decisions, not case-mix
2. **Group 5 reallocated to 3+4 only** — a scar does not change presentation, plurality, or gestational age
3. **Group 9 reference = 100%** — transverse lie cannot deliver vaginally (definitional)

### Main Result

In [ ]:
# These are the CORRECTED 8-category numbers from egypt_gap_v2.py
# Hardcoded from the confirmed final results in PROJECT_MASTER

results = pd.DataFrame([
    {"Hospital": "Benha 2018", "n": 850,
     "Observed": 54.71, "Expected_A": 50.25, "Gap_A": 4.46,
     "Expected_B": 32.70, "Gap_B": 22.01,
     "Inherited": 17.55, "Inherited_pct": 80},
    {"Hospital": "Qena 2022-23", "n": 2186,
     "Observed": 79.69, "Expected_A": 59.04, "Gap_A": 20.65,
     "Expected_B": 38.12, "Gap_B": 41.57,
     "Inherited": 20.92, "Inherited_pct": 50},
])

print("=" * 90)
print("INDIRECT STANDARDISATION — 8-CATEGORY METHOD (CORRECTED)")
print("=" * 90)
print()
print(f"{'':15} {'Observed':>10} {'Expected A':>12} {'Gap A':>8} {'Expected B':>12} {'Gap B':>8} {'Inherited':>10} {'Inh %':>7}")
print("-" * 85)
for _, r in results.iterrows():
    print(f"{r['Hospital']:15} {r['Observed']:>9.2f}% {r['Expected_A']:>11.2f}% "
          f"{r['Gap_A']:>+7.2f} {r['Expected_B']:>11.2f}% {r['Gap_B']:>+7.2f} "
          f"{r['Inherited']:>9.2f} {r['Inherited_pct']:>6.0f}%")

print("""
  Scenario A: Egypt's case-mix as observed — excess from CURRENT practice
  Scenario B: Group 5 set to reference share — TOTAL accumulated excess
  Inherited:  A minus B — excess carried by a case-mix that earlier CS created

  THE DIAGNOSTIC FINDING:
  Benha: 80% of excess is inherited → needs to stop creating new Group 5
  Qena:  50% inherited → needs BOTH to stop new Group 5 AND change current practice
""")

---

## 4. Safety Outcomes by Robson Group

Does a higher CS rate buy better neonatal outcomes?

In [ ]:
safety = load("06_cdc_safety_outcomes_by_group.csv")
if safety is not None:
    print("SAFETY OUTCOMES BY ROBSON GROUP (CDC 2023)")
    print(safety.to_string(index=False))
    print()
    print("  Compare Groups 1 and 3: both spontaneous, term, singleton, cephalic.")
    print("  If a higher CS rate does not show better outcomes,")
    print("  the extra caesareans are not buying safety.")

---

## 5. Practice Patterns — Group 1 (The Cleanest Comparator)

Group 1: nulliparous, term, singleton, cephalic, spontaneous labour. Every woman arrived in labour on her own. No CS was scheduled. Any remaining variation is how labour was managed.

In [ ]:
payer = load("07_group1_payer_standardised.csv")
tod = load("08_group1_time_of_day_standardised.csv")

if payer is not None:
    print("GROUP 1 CS RATE BY PAYER (age-standardised)")
    print(payer.to_string(index=False))
    print()
    print("  After standardisation, Medicaid and Private converge.")
    print("  The crude gap was driven by age, not financial incentive.")
    print()

if tod is not None:
    print("GROUP 1 CS RATE BY TIME OF DAY")
    print(tod.to_string(index=False))

---

## 6. Egypt National CS Rate Trajectory

In [ ]:
egypt = load("09_egypt_national_cs_rates.csv")
if egypt is not None:
    print("EGYPT NATIONAL CS RATES (2005-2026)")
    print()
    for _, row in egypt.iterrows():
        note = ""
        if pd.notna(row.get('note')) and len(str(row.get('note',''))) > 5:
            note = f"  ** {str(row['note'])[:80]}"
        print(f"  {int(row['year'])}  {row['cs_rate_pct']:>5.1f}%  "
              f"{str(row.get('source_type','')):16s}  "
              f"{str(row.get('source',''))[:50]}")
        if note:
            print(f"        {note}")
    print()
    print("  WARNING: 2015 (43.8%) is NOT a CS rate — it is the share of")
    print("  MOTHERS who ever had a CS. Different denominator entirely.")

---

## 7. Predictive Model — Key Results

XGBoost trained on Groups 1, 2a, 3, 4a (narrow population, no class-weighting).

| Metric | Value | Note |
|--------|-------|------|
| ROC-AUC | **0.7991** | Test set (486,820 births); inflated by between-group separation |
| Within-group ROC-AUC | **0.678–0.705** | The honest description |
| AUPRC | 0.3542 | Baseline 0.1172 |
| Brier score | **0.0886** | Calibrated: predicted 11.70% vs observed 11.72% |

### The calibration correction story

| | With class-weighting | **Without (corrected)** |
|---|---|---|
| Brier | 0.1857 | **0.0886** |
| Mean predicted | ~39.4% | **11.70%** |
| ROC-AUC | 0.7985 | **0.7985** (unchanged) |

`scale_pos_weight=7.6` was inflating every probability. Removing it fixed calibration with no loss in discrimination.

In [ ]:
perf = load("dash_model_performance.csv")
byg = load("dash_by_group.csv")
if byg is None:
    byg = load("test_results_by_group.csv")

if perf is not None:
    print("MODEL PERFORMANCE — VALIDATION vs TEST")
    print(perf.to_string(index=False))
    print()

if byg is not None:
    print("PERFORMANCE BY ROBSON GROUP (test set)")
    print(byg.to_string(index=False))
    print()
    print("  The within-group range is the honest description of model skill.")
    print("  Much of the headline ROC-AUC comes from separating groups that")
    print("  sit at very different CS rates (Group 3 ≈ 2.5%, Group 2a ≈ 29%).")

---

## 8. What Drives the Model (SHAP)

SHAP explains THE MODEL, not the medicine. It reports which inputs the model relied on — including patterns driven by practice rather than physiology.

In [ ]:
imp = load("shap_importance.csv")
if imp is None:
    imp = load("dash_feature_importance.csv")

if imp is not None:
    print("SHAP FEATURE IMPORTANCE (top 15)")
    top = imp.head(15)
    cols = [c for c in ['feature', 'share_%', 'overall_share_%',
                        'within_group_share_%', 'difference'] if c in top.columns]
    print(top[cols].to_string(index=False))
    print()
    print("  Anthropometric + demographic ≈ 77% of model decisions")
    print("  Medical risk factors ≈ 6%")
    print()
    print("  Among women with no scar, at term, single cephalic fetus,")
    print("  documented medical indications explain very little of who")
    print("  ends up with a caesarean.")

---

## 9. Conclusions

### The central finding

Egypt's caesarean excess over the reference is **real, measurable, and robust** across every sensitivity test:

- **Benha:** +4.46 pp above expected (80% inherited from earlier CS)
- **Qena:** +20.65 pp above expected (50% inherited, 50% current practice)

### The model confirms from a different angle

Within fixed Robson groups, the measurable clinical picture accounts for within-group ROC-AUC of only 0.678–0.705. The residue — what the clinical variables cannot explain — is where practice lives.

### Different interventions needed

| Hospital | Priority |
|----------|----------|
| **Benha** | Stop new Group 5 from forming → reduce CS in first births |
| **Qena** | That, PLUS change current practice across all groups |

### Self-corrections as strengths

| What was wrong | What was fixed | Effect |
|----------------|---------------|--------|
| 10-category standardisation | Merged to 8 (onset = decision) | Benha gap: 7.54 → **4.46** pp |
| class-weighting (spw=7.6) | Removed | Brier: 0.1857 → **0.0886** |
| F2 threshold | Withdrawn (predicts "performed" not "needed") | Prevents manufactured prediction |
| Group 10 scenario | Withdrawn (no clean reallocation) | Reduces unfounded claims |
| Group 5 proportional reallocation | To 3+4 only (derivation from definitions) | Old method misplaced 6.6% of births |
| Within-group SHAP metric: mean(\|SHAP\|) | Changed to std(SHAP) | PRIORLIVE dropped from 23% to near-zero in Group 1 (was constant, not discriminating) |

Every correction **reduced** the estimate, **removed** a claim, or **fixed a misleading metric**. The project is more conservative than its earlier versions, and every change has a documented reason.

---

### Notebooks for the full methodology

| Notebook | Contents |
|----------|----------|
| 01 | Robson classifier + 21 tests + CDC pipeline + data dictionary |
| 02 | Validation + Group 9 + VBAC + practice patterns + safety |
| 03 | DHS/EHIS + standardisation + **8-category gap** + cost + governorates |
| 04 | Model pipeline + **calibration fix** + threshold + evaluation |
| 05 | SHAP + temporal validation 2024 + kappa audit tool |
| 06 | Power BI export + model card + project organisation |

---
*End of MASTER notebook*